# publish_gold — Gold layer (R5/R7), built OUTSIDE the SDP pipeline

**Why this exists (R4):** the gold tables are no longer pipeline datasets. The `claims_daily` job runs them
via this notebook as the `publish_gold` task, only after the `dq_gate` if/else task passes
(`dq_problems == 0`). The pipeline now covers bronze + silver only.

**What it builds:**
- `claims.gold.claims_enriched` — one row per claim, exact R5 column contract (31 cols), CLUSTER BY (admission_month, hospital_id)
- `claims.gold.hospital_monthly` — monthly per-hospital aggregates

**Sources:** `claims.silver.claims_clean`, `claims.silver.dim_policy_scd2`, `claims.silver.plans`,
`claims.silver.hospitals`, `claims.silver.hospital_network_history` (pipeline-published tables, read with qualified names).

⚠️ Cell 2 **drops the legacy pipeline-owned materialized views** of the same names (one-time migration;
both are fully recomputable from silver). After the rebuild, the R1 row filter is re-attached in cell 5.
NOTE: R2 column comments on the old gold MVs are lost in this migration and would need re-applying if required.

In [0]:
# One-time migration: remove the legacy pipeline-owned materialized views so the
# batch tables below can be created with the same names. Both are fully recomputable
# from claims.silver.*, so no data is lost. Re-running this cell is safe (IF EXISTS).
dbutils.widgets.text("catalog", "claims")   # target catalog (bundle job passes dev/prod catalog)
catalog = dbutils.widgets.get("catalog")

for name in [f"{catalog}.gold.claims_enriched", f"{catalog}.gold.hospital_monthly"]:
    for stmt in (f"DROP MATERIALIZED VIEW IF EXISTS {name}",
                 f"DROP VIEW IF EXISTS {name}",
                 f"DROP TABLE IF EXISTS {name}"):
        try:
            spark.sql(stmt)
        except Exception:
            pass  # object was not of that type
print("Legacy gold MVs dropped.")

In [0]:
%sql
-- R5/R7: claims_enriched — exact column contract (31 cols), no adjudication columns
-- (payable / eligible / waiting period / copay) by design. Clustered by (admission_month, hospital_id).
CREATE OR REPLACE TABLE ${catalog}.gold.claims_enriched
COMMENT 'R5/R7: one row per claim, enriched with policy/plan/hospital/network attributes as of admission date (batch-built by publish_gold notebook, claims_daily job)'
CLUSTER BY (admission_month, hospital_id)
AS
WITH policy_at_admission AS (
  -- Point-in-time policy version in force at each claim's admission date (SCD2 as-of join)
  SELECT claim_id, plan_code
  FROM (
    SELECT
      c.claim_id,
      pv.plan_code,
      ROW_NUMBER() OVER (
        PARTITION BY c.claim_id
        ORDER BY pv.effective_from DESC
      ) AS rn
    FROM ${catalog}.silver.claims_clean c
    JOIN ${catalog}.silver.dim_policy_scd2 pv
      ON c.policy_id = pv.policy_id
     AND pv.__START_AT <= c.admission_date
     AND (pv.__END_AT IS NULL OR pv.__END_AT > c.admission_date)
  )
  WHERE rn = 1
),
policy_start_dates AS (
  -- Original policy start date (earliest effective_from across all versions)
  SELECT
    policy_id,
    MIN(effective_from) AS policy_start_date
  FROM ${catalog}.silver.dim_policy_scd2
  GROUP BY policy_id
),
network_at_admission AS (
  -- Network status of the claim's hospital at the admission date; no coverage => FALSE (R5)
  SELECT claim_id, TRUE AS is_network
  FROM (
    SELECT
      c.claim_id,
      ROW_NUMBER() OVER (
        PARTITION BY c.claim_id
        ORDER BY nh.network_from DESC
      ) AS rn
    FROM ${catalog}.silver.claims_clean c
    JOIN ${catalog}.silver.hospital_network_history nh
      ON c.hospital_id = nh.hospital_id
     AND nh.network_from <= c.admission_date
     AND (nh.network_to IS NULL OR nh.network_to >= c.admission_date)
  )
  WHERE rn = 1
)
SELECT
  c.claim_id,
  c.member_id,
  c.policy_id,
  c.hospital_id,
  c.admission_date,
  c.discharge_date,
  CAST(date_trunc('month', c.admission_date) AS DATE) AS admission_month,
  c.diagnosis_code,
  c.procedure_code,
  c.is_pre_existing,
  c.claim_type,
  c.billed_amount,
  c.room_rent_per_day,
  c.room_days,
  c.status,
  c.approved_amount,
  c.rejection_code,
  c.submitted_ts,
  c.settled_ts,
  c.channel,
  pa.plan_code AS plan_code_at_admission,
  pl.plan_family,
  pl.room_rent_cap_per_day,
  ps.policy_start_date,
  CAST(FLOOR(months_between(c.admission_date, ps.policy_start_date)) AS INT)
    AS policy_tenure_months,
  CASE
    WHEN c.discharge_date IS NULL THEN NULL
    ELSE GREATEST(datediff(c.discharge_date, c.admission_date), 1)  -- same-day stay = 1 day (Q-E convention)
  END AS length_of_stay,
  CASE
    WHEN pl.room_rent_cap_per_day IS NULL THEN NULL
    ELSE c.room_rent_per_day > pl.room_rent_cap_per_day
  END AS room_rent_over_cap,
  CASE
    WHEN c.settled_ts IS NOT NULL AND c.submitted_ts IS NOT NULL
      THEN datediff(c.settled_ts, c.submitted_ts)
  END AS turnaround_days,  -- empty until settled
  COALESCE(nw.is_network, FALSE) AS is_network_at_admission,  -- never empty (R5)
  h.city AS hospital_city,
  h.city_tier AS hospital_city_tier
FROM ${catalog}.silver.claims_clean c
LEFT JOIN policy_at_admission pa ON pa.claim_id = c.claim_id
LEFT JOIN ${catalog}.silver.plans pl ON pl.plan_code = pa.plan_code
LEFT JOIN policy_start_dates ps ON ps.policy_id = c.policy_id
LEFT JOIN network_at_admission nw ON nw.claim_id = c.claim_id
LEFT JOIN ${catalog}.silver.hospitals h ON h.hospital_id = c.hospital_id;

In [0]:
%sql
-- R5: hospital_monthly — monthly per-hospital aggregates. Money as DECIMAL(2dp);
-- totals never empty (0 when no approved amounts); counts as INT.
CREATE OR REPLACE TABLE ${catalog}.gold.hospital_monthly
COMMENT 'R5: monthly claim volume and amounts per hospital (batch-built by publish_gold notebook, claims_daily job)'
AS
SELECT
  e.hospital_id,
  e.admission_month,
  CAST(COUNT(*) AS INT) AS claim_count,
  CAST(COALESCE(SUM(e.billed_amount), 0.0) AS DECIMAL(15,2)) AS total_billed,
  CAST(COALESCE(SUM(e.approved_amount), 0.0) AS DECIMAL(15,2)) AS total_approved,
  CAST(COUNT(CASE WHEN e.status = 'rejected' THEN 1 END) AS INT) AS rejected_count
FROM ${catalog}.gold.claims_enriched e
GROUP BY e.hospital_id, e.admission_month;

In [0]:
# Re-attach the R1 governance row filter (main `claims` catalog only — the filter
# function lives in claims.gov), then verify row counts and gold shape.
if catalog == "claims":
    spark.sql("""
      ALTER TABLE claims.gold.claims_enriched
      SET ROW FILTER claims.gov.claims_state_filter ON (hospital_id)
    """)
else:
    print(f"catalog={catalog}: skipped R1 row filter (claims.gov.claims_state_filter is main-catalog governance)")

verify = spark.sql(f"""
  SELECT
    (SELECT COUNT(*) FROM {catalog}.gold.claims_enriched)  AS claims_enriched_rows,
    (SELECT COUNT(*) FROM {catalog}.gold.hospital_monthly) AS hospital_monthly_rows,
    (SELECT COUNT(DISTINCT claim_id) FROM {catalog}.gold.claims_enriched) AS distinct_claims,
    (SELECT COUNT(*) FROM {catalog}.information_schema.columns
      WHERE table_schema = 'gold' AND table_name = 'claims_enriched') AS column_count,
    (SELECT COUNT(*) FROM {catalog}.information_schema.row_filters
      WHERE table_schema = 'gold' AND table_name = 'claims_enriched') AS row_filters
""")
display(verify)